# பாடம் 13 - கோக்னி நுண்ணறிவு வரைபடங்களுடன் முகவர் நினைவகம்


## அமைப்பு

இந்த நோட்டுபுக் எவ்வாறு [**Cognee**](https://www.cognee.ai/) அறிவு வரைபடங்கள் மற்றும் **Microsoft Agent Framework** (MAF) பயன்படுத்தி நினைவிடத்தை நிரந்தரமாகக் கொண்ட புத்திசாலியான **கோடிங் உதவியாளர்** உருவாக்குவது என்பதை விளக்குகிறது.

Cognee அமைப்பு இல்லாத உரையை குறிச்சொல் சார்ந்த, வினவக்கூடிய அறிவு வரைபடமாக மாற்றி, வெக்டர் எழுத்துகளை ஆதரவு செய்து — உங்கள் முகவரிக்கு வளமான, தொடர்பு உணர்வுள்ள நீண்டகால நினைவிடத்தைக் கொடுக்கிறது.

### நீங்கள் கற்கப்போகும் விஷயங்கள்
1. **அறிவு வரைபடங்களை உருவாக்குதல்**: டெவலப்பர் சுயவிவரங்கள் மற்றும் சிறந்த பயிற்சிகளை அமைப்பு கொண்ட, வினவக்கூடிய அறிவாக மாற்றுதல்.
2. **Cognee-ஐ MAF உடன் இணைத்தல்**: `@tool` செயல்பாடுகளைப் பயன்படுத்தி ஒரு MAF முகவரி Cognee-வின் அறிவு வரைபடத்தை வினவ அனுமதித்தல்.
3. **அமர்வு உணர்ந்த உரையாடல்கள்**: ஒரே அமர்வில் பல கேள்விகளுக்கு இடையேயான சூழலை பாதுகாத்தல்.
4. **நீண்டகால நினைவிடம்**: முக்கிய அறிவை அமர்வுகளுக்கு மابين நிரந்தரமாக வைத்துக் கொண்டு புதிய உரையாடல்களில் மீட்டெடுத்தல்.

### முன்பக்கங்கள்
- Python 3.9+
- அமர்வு மேலாண்மைக்காக உள்ளூர் ரெடிஸ் இயங்கு நிலையில் (`docker run -d -p 6379:6379 redis`)
- ஒரு LLM API விசை (எ.கா. OpenAI) — `.env` இல் `LLM_API_KEY` அமைக்கப்பட வேண்டும்
- `.env` இல் `CACHING=true` (Cognee அமர்வுகளுக்கு தேவையானது)
- Azure AI Foundry திட்டம் மற்றும் அதில் chat மாதிரி ஒப்படைக்கப்பட்டது
- `.env` இல் `AZURE_AI_PROJECT_ENDPOINT` மற்றும் `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI அங்கீகரிக்கப்பட்டது (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## முகவர் நினைவக வகைகள்

இந்த நோட்புக் முக்கியமான பாடம் 13 நோட்புக் உள்ள அதே மூன்று நினைவக வகைகளை ஆராய்கிறது, ஆனால் Cognee-ஐ நீண்டகால நினைவக பின்புலமாக பயன்படுத்துகிறது:

| நினைவக வகை | செயல்முறை | ஆயுள் காலம் |
|---|---|---|
| **செயற்கை** | `agent.create_session()` (MAF) | ஒரே உரையாடல் |
| **குறுகிய கால** | Cognee session cache (Redis) | ஒரே சஸ்சன் |
| **நீண்டகால** | Cognee வலை அறிதல் வரைபடம் + வெக்டர்கள் | நிரந்தரமானது |

### Cognee-என் நினைவக அமைப்பு
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## கோக்னி சேமிப்பகத்தை தயார் செய்யுங்கள்


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## பகுதி 1 — அறிவுத்தளத்தை உருவாக்குதல்

எமது குறியீட்டு உதவியாளருக்கான விரிவான அறிவுத்தளத்தை உருவாக்க மூன்று வகையான தரவுகளை நாம் இறக்குமதி செய்கிறோம்:

1. **முன்னணி வடிவமைப்பாளர் சுயவிவரம்** — தனிப்பட்ட நிபுணத்துவம் மற்றும் தொழில்நுட்ப பின்னணி  
2. **Python சிறந்த நடைமுறைகள்** — நடைமுறை வழிகாட்டுதல்களுடன் Python இன் சென்  
3. **வரலாற்றுச் சந்திப்புகள்** — முன்னைய கேள்வி மற்றும் பதில் அமர்வுகள் முன்னணி வடிவமைப்பாளர்கள் மற்றும் AI உதவியாளர்களுக்கு இடையிலான


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## அறிவியல் குறியீட்டு வரைபடத்தை காண்பிக்கவும்

Cognee எடுக்கப்பட்ட பொருட்கள் மற்றும் தொடர்புகளின் இடையே தொடர்புடைய HTML காட்சிப்படுத்தலை உருவாக்க முடியும்.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memify உடன் நினைவாற்றலை வளப்படுத்துங்கள்

`memify()` அறிவுக் கிராப் பகுப்பாய்வு செய்து புத்திசாலி விதிகளைக் உருவாக்குகிறது — முறைகள், சிறந்த நடைமுறைகள் மற்றும் கருத்துக்களின் இடையிலான உறவுகளை கண்டறிகிறது.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## பகுதி 2 — Cognee கருவிகளுடன் MAF முகவர்

இப்போது நாம் `@tool` செயல்பாடுகள் மூலம் Cognee இன் அறிவுக் கோவையை விசாரிக்கக்கூடிய MAF முகவர்களை உருவாக்குகிறோம். இது முகவருக்கு உரையாடல் சூழலை sessões மூலம் பராமரிக்கும்போது, புலமை அறிவியல் தேடலின் முழு சக்தியையும் பயன்படுத்த உதவுகிறது.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## அமர்வுகளுடன் செயலில் நினைவகம்

`AgentSession` (`agent.create_session()` மூலம் உருவாக்கப்படுகிறது) ஒரு அமர்வின் உள்ளே செயலில் நினைவகத்தை வழங்குகிறது. முகவர் முன்னைய செய்திகளை மறுபடியும் பார்க்கவும், மேலும் Cognee இன் நீண்டகால அறிவு வரைபடத்தைக் கேள்வி செய்வதும் முடியும்.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## புதிய அமர்வு — நீண்டகால நினைவிடம் நிலைத்திருக்கிறது

புதிய அமர்வு துவங்குவதால் வேலை செய்யும் நினைவகம் சுத்தப்படுத்தப்படுகின்றது, ஆனால் Cognee அறிவு கிராஃப் இன்னும் கிடைக்கும். வழிகாட்டி புதிய உரையாடலில் கூட அந்த நீண்டகால அறிவைப் பெற முடியும்.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## சாராம்சம்

இந்த நோட்புக்கில் நீங்கள் **MAF-இன் வேலை நினைவகத்தை** (`agent.create_session()`) **Cognee-இன் நீண்டகால அறிவுக் கண்ணியப்போட்டியுடன்** சேர்க்கும் ஒரு குறியீட்டு உதவியாளரை கட்டியுள்ளீர்கள்.

### நீங்கள் கற்றுக்கொண்டது
1. **அறிவுக் கண்ணியப்போட்டிய உருவாக்கம்**: Cognee கட்டமைக்காத உரையை அளித்து, ஒரு கண்ணியப்போட்டி மற்றும் வெக்டர் நினைவகத்தை உருவாக்குகிறது.
2. **memify மூலம் கண்ணியப்போட்டியின் வளப்படுத்தல்**: உங்கள் உள்ளகக் கண்ணியப்போட்டியின் மேல் தொழில்நுட்பத் தகவல்கள் மற்றும் வளமான உறவுகள்.
3. **MAF + Cognee ஒருங்கிணைவு**: `@tool` செயல்பாடுகள் MAF முகவரிக்கு Cognee-இன் கண்ணியப்போட்டியை இயல்பாக கேள்வி செய்ய அனுமதிக்கிறது.
4. **வேலை நினைவகம் + நீண்டகால நினைவகம்**: `AgentSession` (`agent.create_session()` மூலம்) அமர்வு சூழலை வழங்கும் போது Cognee நிலையான அறிவை வழங்குகிறது.
5. **NodeSets மூலம் வடிகட்டப்பட்ட தேடல்**: அறிவுக் கண்ணியப்போட்டியின் குறிப்பிட்ட பகுதிகளை இலக்கு செய்ய (எ.கா. கோட்பாடுகள் மட்டும்).

### முக்கிய கற்றுக் கொள்கைகள்
- **Cognee** திட வெக்டர் நினைவகம் விட சக்திவாய்ந்த, கட்டமைக்கப்பட்ட, உறவுகளை உணர்ந்து நினைவகமாக்குகிறது.
- **`@tool` செயல்பாடுகள்** MAF முகவர்களும் மாறும் அறிவுத் தொகுதிகளும் இடையே தெளிவான பாலமாகும்.
- **`AgentSession`** (`agent.create_session()` மூலம்) ஒவ்வொரு உரையாடலுக்கும் தனியான சூழலை நீண்டகால அறிவிலிருந்து பிரித்து பராமரிக்கிறது.
- ஒருเดียวய்க் கண்ணியப்போட்டி பல அமர்வுகளுக்கும் முகவர்களுக்கும் பயன்படுகிறது.

### இயல்பான பயன்பாடுகள்
- **ஆசிரியர் கூடுதல் உதவியாளர்கள்**: குறியீட்டு பரிசீலனை, சம்பவ பகுப்பாய்வு, கட்டமைப்பு உதவியாளர்கள்
- **வாடிக்கையாளர் எதிர் கூடுதல் உதவியாளர்கள்**: தயாரிப்பு ஆவணங்கள், அடிக்கடி கேட்கப்படும் கேள்விகள் மற்றும் CRM குறிப்புகளை முன்நிலைப்படுத்தும் ஆதரவு முகவர்கள்
- **ஆன்டார்நல் நிபுணர் கூடுதல் உதவியாளர்கள்**: கொள்கைகள், சட்டம் அல்லது பாதுகாப்பு உதவியாளர்கள் வழிகாட்டுதல்களுக்கு ஏற்ப உடனடி கருத்துபரிமாற்றம்
- **ஐக்கிய தரவு அடுக்குகள்**: கட்டமைக்கப்பட்ட மற்றும் கட்டமைக்காத தரவை ஒரே கேள்விக்கிடைக்கும் கண்ணியப்போட்டியாக சேகரிக்கும்

### அடுத்த படிகள்
- Cognee இல் கால சார்ந்த அறிவை சோதனை செய்
- துறை சார்ந்த கண்ணியப்போட்டியின் தரத்தை நிர்ணயிக்கும் OWL அறிவியலை வரையறு
- பெறுவானசொற்றை நேர்த்திசெய்ய பயனர் பின்னூட்டத்தைச் சேர்க்கவும்
- ஒரே Cognee நினைவக அடுக்கு பகிரும் பல முகவரிகள் கொண்ட அமைப்புகளுக்கு அளவை விரிவாக்கு


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**உறுதிப்பத்திரம்**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டது. எங்களின் துல்லியத்தன்மைக்கு முயற்சி செய்கிறோம் என்றாலும், தானியங்கி மொழிபெயர்ப்புகளில் சில தவறுகள் அல்லது துல்லியக்குறைவுகள் இருக்கக்கூடும் என்பதை கவனத்தில் கொள்ளவும். உள்ளூர் மொழியில் உள்ள அசிலிய ஆவணம் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பை பரிந்துரைக்கின்றோம். இந்த மொழிபெயர்ப்பை பயன்படுத்துவதால் ஏற்படும் எந்தவொரு தவறான புரிதல் அல்லது தவறான விளக்கத்திற்கு எங்களை பொறுப்புற்றவனாக்க முடியாது.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
